# Análise exploratória: 2023–2024

Este caderno examina a distribuição da alfabetização, a cobertura e os padrões municipais em 2023 e 2024. As partições individuais e os indicadores municipais são limitados a esses anos; a avaliação de 2025 é tratada nos cadernos de resultados.

Defina `FASE3_MARTS` com a pasta dos conjuntos preparados e `FASE3_EDA_OUTPUT` com uma pasta de saída fora do conjunto de origem. São utilizados `alunos_2023.parquet`, `alunos_2024.parquet`, `municipios.parquet` e o `coverage.json` do diretório pai.

In [ ]:
import os
import sys
from pathlib import Path
for candidate in (Path.cwd() / 'src', Path.cwd().parent / 'src'):
    if candidate.is_dir():
        sys.path.insert(0, str(candidate))
        break
import pandas as pd
from IPython.display import Image, display
from alfabetizacao_ml.eda import run_eda
if not os.environ.get('FASE3_MARTS') or not os.environ.get('FASE3_EDA_OUTPUT'):
    raise RuntimeError('Defina FASE3_MARTS e FASE3_EDA_OUTPUT para caminhos privados.')
marts_dir = Path(os.environ['FASE3_MARTS'])
output_dir = Path(os.environ['FASE3_EDA_OUTPUT'])
report = run_eda(marts_dir, output_dir)
assert report['development_years'] == [2023, 2024]
assert not report['test_labels_used'] and not report['test_metrics_used']
print('EDA concluída somente para 2023–2024; saídas agregadas privadas.')

## População e cobertura

A população elegível compreende o segundo ano das redes estadual e municipal, com rótulo oficial binário e peso positivo finito. As exclusões são contadas por motivo, começando pelas informações de identificação indispensáveis; cada registro pertence a uma única categoria. A falta de covariáveis históricas é preservada para tratamento posterior.

Uma UF ou rede ausente da fonte não corresponde a taxa zero. Os resultados descrevem a população efetivamente coberta.

In [ ]:
coverage = pd.DataFrame(report['coverage']).T
coverage.index.name = 'ano'
display(coverage)
display(pd.DataFrame(report['students']['year']))
display(pd.DataFrame(report['students']['year_uf_network']))

## Percentuais ponderados de alfabetização

Os percentuais são calculados por `100 × Σ(peso × indicador) / Σ(peso)`. Alfabetização e não alfabetização são complementares na população elegível. A comparação com os percentuais não ponderados permite observar a influência dos pesos. Taxas de grupos com menos de 30 alunos são suprimidas.

In [ ]:
display(pd.DataFrame(report['students']['year_network']))
for filename in ('01_weighted_network.png', '02_weighted_uf.png'):
    display(Image(filename=str(output_dir / filename)))

## Disponibilidade do contexto anterior

A série AEEB utilizada começa em 2023, portanto suas observações desse ano não dispõem de contexto de 2022. Em 2024, ausências podem decorrer da cobertura do ano anterior. INSE e Censo referem-se a 2021 e 2022, respectivamente.

A ausência ponderada expressa a massa de alunos afetada; nas contagens municipais, cada município tem o mesmo peso.

In [ ]:
display(pd.DataFrame(report['student_missing']))
display(pd.DataFrame(report['municipal']['missing']))
display(Image(filename=str(output_dir / '03_context_missing.png')))

## Distribuições e associações municipais

Cada município recebe peso igual. Pearson descreve associação linear, e Spearman, associação monotônica. São informados os pares completos disponíveis por ano e variável; falta de variação ou de pares suficientes resulta em medida indisponível.

Associações municipais não identificam causalidade nem descrevem automaticamente alunos ou famílias. A repetição de um contexto entre alunos também não cria observações independentes.

In [ ]:
display(pd.DataFrame(report['municipal']['correlations']))
for filename in ('04_municipal_distribution.png', '05_context_correlations.png'):
    display(Image(filename=str(output_dir / filename)))

## Mudanças entre 2023 e 2024

O painel compara os mesmos municípios da rede municipal nos dois anos e contabiliza os que ficam fora da comparação. Ele não acompanha necessariamente os mesmos alunos. Mudanças de composição, cobertura e revisões da fonte podem contribuir para as diferenças observadas. Duas observações anuais são insuficientes para estabelecer uma tendência de longo prazo.

In [ ]:
display(pd.DataFrame([report['municipal']['balanced_panel']]))
display(pd.DataFrame(report['municipal']['changes_by_uf']))
if '06_balanced_changes.png' in report['figures']:
    display(Image(filename=str(output_dir / '06_balanced_changes.png')))

## Metas municipais de 2024

Os resultados de 2024 são comparados às metas da edição de referência de 2024. O denominador reúne municípios com resultado e meta avaliáveis; metas ausentes são informadas separadamente. O operador `>80` exige valor superior a 80, enquanto `>=80` admite igualdade. Essa comparação não inclui 2023, e edições posteriores das metas podem apresentar revisões.

In [ ]:
display(pd.DataFrame([report['municipal']['targets_2024_total']]))
display(pd.DataFrame(report['municipal']['targets_2024_by_uf']))
display(Image(filename=str(output_dir / '07_targets_2024.png')))

## Hipóteses e cuidados para a modelagem

1. O histórico municipal pode estar associado ao resultado posterior e ao risco contextual.
2. O INSE pode estar associado aos resultados municipais, sem diagnosticar habilidades individuais ou características familiares.
3. Diferenças entre UF e rede podem refletir composição, cobertura e condições estruturais.

Essas hipóteses são observacionais. O desenvolvimento de 2024 utiliza contexto de 2023, com municípios separados entre validação e reserva. Imputação e transformações são ajustadas no treino de cada etapa. Identificadores, proficiência contemporânea, respostas, presença e agregados do próprio ano ficam fora dos atributos preditivos.

In [ ]:
assert all(set(item['allowed_years']).issubset({2023, 2024}) for item in report['input_audit'])
display(pd.DataFrame(report['input_audit']))
print('Artefatos agregados:', ', '.join(report['figures']), 'e eda_summary.json')